# Lesson 8.1 · Bagging and forests

In module 7 you saw that a deep tree has **low bias and high variance**: it changes a lot if the sample changes a little. A random forest exploits this flaw: it trains many deep trees on different samples and **averages them**. The average of estimators
that are variable but little biased has a much smaller variance, without increasing the bias.

**What you will learn**

- **Bagging** (bootstrap aggregating): sampling with replacement and voting.
- What the random forest adds: a random subset of features at each split (`max_features`), which **decorrelates** the trees.
- The **out-of-bag** sample and the OOB error: a free validation estimate.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
forest = RandomForestClassifier(200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
for name, m in {"one tree": tree, "bagging of 200 trees": bagging, "random forest of 200 trees": forest}.items():
    print(f"{name:28s} train {m.score(X_tr, y_tr):.3f} | test {m.score(X_te, y_te):.3f}")

## The bootstrap and the vote

A **bootstrap** sample is a sample with replacement of the same size as the training set: it contains about **63%** of the distinct examples (each one appears zero, one or more times). Each tree is trained on a different sample; the final prediction is the **majority vote**
(in scikit-learn, the average of the leaf probabilities). By averaging, the "random" errors of each tree (those due to its sample) cancel out, and the systematic ones remain.

In [ ]:
rng = np.random.default_rng(0)
n = 2000
share = np.mean([len(np.unique(rng.integers(0, n, n))) / n for _ in range(20)])
print(f"average share of distinct examples in a bootstrap sample: {share:.3f} (theoretical 1 - 1/e = {1 - 1 / np.e:.3f})")

## What the random forest adds

Bagging has a limit: the trees are **correlated** (if a feature is very strong, they all choose it at the root and look alike). The variance of the average of $B$ variables with variance $\sigma^2$ and correlation $\rho$ is

$$\rho\,\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

Adding trees only reduces the second term: the first, $\rho\sigma^2$, stays. The forest reduces $\rho$ by choosing, at **every split**, the best among a random subset of features (`max_features`, by default $\sqrt{p}$ in classification): more diverse trees, a better average.

> **Common mistake.** Thinking that "more trees = more overfitting". Adding trees to a forest does not increase overfitting: the variance drops and levels off. Overfitting depends on the complexity of the individual trees and on their correlation.

## The out-of-bag error

Each tree does not see about 37% of the examples (the ones outside its bootstrap). Predicting each example **only with the trees that did not see it** gives an honest prediction for the whole training set: the **OOB error**, a free estimate of generalization, with no validation set.

In [ ]:
oob = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"OOB accuracy: {oob.oob_score_:.3f} | accuracy on the test set: {oob.score(X_te, y_te):.3f} | (on the training set: {oob.score(X_tr, y_tr):.3f})")

The OOB accuracy is close to the test one, while the training one is 1.0 (each tree has seen 63% of the examples and the forest memorizes them). **Never evaluate a forest on the training set**: the OOB is the correct internal estimate.

## Check your understanding

<details><summary>Why does the bootstrap sample contain about 63% of distinct examples?</summary>

Because the probability that an example is never drawn in n draws with replacement is (1 − 1/n)^n → 1/e ≈ 0.368: 36.8% stays out (out-of-bag) and 63.2% is present at least once.

</details>

<details><summary>Why does a small max_features improve the forest?</summary>

Because it reduces the correlation between the trees: if every split can only use some of the features, the trees do not all choose the same dominant feature at the root, and the average of less correlated trees has a smaller variance. If it is too small, though, each tree is weak (more bias): it is a hyperparameter to choose.

</details>